# 00a Python Essentials

This is a quick refresher on the Python features you'll use most when working with JAX. If you're comfortable with functions, classes, decorators, and unpacking, feel free to skip ahead to [Chapter 01](../01%20Just-In-Time%20Compilation.ipynb).

## Functions as first-class objects

JAX transformations like `jit`, `grad`, `vmap` take functions as input and return new functions.

In [ ]:
def square(x):
    return x ** 2

# Functions are values — assign, pass, return
apply = square
print(apply(5))  # 25

# Lambda for inline functions
double = lambda x: x * 2
print(double(5))  # 10

# Higher-order function (takes a function as argument)
def apply_twice(f, x):
    return f(f(x))

print(apply_twice(square, 3))  # 81

## Decorators and `functools.partial`

JAX uses decorators everywhere: `@jax.jit`, `@jax.grad`, `@partial(jax.jit, static_argnames=...)`.

In [ ]:
from functools import partial

def log_calls(f):
    def wrapper(*args, **kwargs):
        print(f"Calling {f.__name__}")
        return f(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    return a + b

print(add(2, 3))

# partial fixes some arguments
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)
print(square(5), cube(2))  # 25 8

## Unpacking and starred expressions

Pytree operations and layer iteration use these heavily.

In [ ]:
# Tuple unpacking
a, b = (1, 2)

# Star unpacking — very common in JAX for separating layers
*hidden, last = [10, 20, 30, 40]
print(f"hidden={hidden}, last={last}")  # hidden=[10, 20, 30], last=40

# Works with any iterable
first, *middle, last = range(5)
print(f"first={first}, middle={middle}, last={last}")

# Dict unpacking
defaults = {"lr": 0.01, "epochs": 100}
overrides = {"lr": 0.001}
config = {**defaults, **overrides}
print(config)  # {'lr': 0.001, 'epochs': 100}

## Type hints and `NamedTuple`

JAX uses NamedTuples as parameter containers and pytree nodes.

In [ ]:
from typing import NamedTuple
import numpy as np

class Params(NamedTuple):
    weights: np.ndarray
    bias: np.ndarray

def init(n_in: int, n_out: int) -> Params:
    return Params(
        weights=np.random.randn(n_in, n_out),
        bias=np.zeros(n_out),
    )

p = init(3, 2)
print(f"weights shape: {p.weights.shape}, bias shape: {p.bias.shape}")
print(f"Accessed by name: {p.weights.shape}")
print(f"Accessed by index: {p[0].shape}")  # NamedTuples are tuples

## Comprehensions and generators

In [ ]:
# List comprehension
squares = [x ** 2 for x in range(5)]
print(squares)

# Dict comprehension
layer_sizes = {"layer_0": (3, 4), "layer_1": (4, 2)}
param_counts = {k: v[0] * v[1] for k, v in layer_sizes.items()}
print(param_counts)

# Generator with iter/next (used for PRNG key iteration in JAX)
keys = iter(range(10))
first = next(keys)
second = next(keys)
print(f"first={first}, second={second}")

## Classes

Custom pytree nodes require class registration.

In [ ]:
class Layer:
    def __init__(self, n_in, n_out):
        self.weights = np.random.randn(n_in, n_out)
        self.bias = np.zeros(n_out)

    def __repr__(self):
        return f"Layer(weights={self.weights.shape}, bias={self.bias.shape})"

layers = [Layer(3, 4), Layer(4, 2)]
for layer in layers:
    print(layer)

## Summary

| Python Feature | Where it appears in JAX |
|---|---|
| Functions as values | `jax.jit(f)`, `jax.grad(f)`, `jax.vmap(f)` |
| Decorators | `@jax.jit`, `@jax.grad` |
| `functools.partial` | `@partial(jax.jit, static_argnames=...)` |
| Star unpacking | `*hidden, last = layers` in forward passes |
| `NamedTuple` | Parameter containers, pytree nodes |
| Comprehensions | Building parameter lists, layer initialization |
| Classes + `__repr__` | Custom pytree node registration |

## Exercises

**Exercise 1:** Write a higher-order function `compose(f, g)` that returns a new function computing `f(g(x))`. Test it by composing `square` and `double`.

In [ ]:
# Your code here

**Exercise 2:** Create a `LayerParams` NamedTuple with `weights` and `bias` fields. Write a function that takes a list of `LayerParams` and returns a dict mapping layer index to total parameter count.

In [ ]:
# Your code here

**Exercise 3:** Write a decorator `@timer` that prints how long a function takes to execute (use `time.time()`). Apply it to a function that sums a large range.

In [ ]:
# Your code here